In [0]:
from pyspark.sql import functions as F

cat = "capstone6_telecom"

# source table in Azure SQL -> Bronze table, and the row count in the seed file
masters = {
    "tower_master":            ("bronze_tower_master",            161),
    "network_topology_master": ("bronze_network_topology_master", 170),
    "region_lookup":           ("bronze_region_lookup",           179),
}

In [0]:
server   = "capstone6dbserver.database.windows.net"
database = "capstone6db"
user     = dbutils.secrets.get("azure-sql-scope", "sql-user")
password = dbutils.secrets.get("azure-sql-scope", "sql-pwd")

jdbc_url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;")

def read_master(src):
    return (spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"dbo.{src}")
            .option("user", user)
            .option("password", password)
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load())

## Load into Bronze

In [0]:
for src, (target, expected) in masters.items():
    df = read_master(src)
    cols = df.columns
    out = (df
           .withColumn("source_system", F.lit(f"azure_sql.dbo.{src}"))
           .withColumn("ingest_ts", F.current_timestamp())
           .withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]), 256)))
    out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.bronze.{target}")
    n = spark.table(f"{cat}.bronze.{target}").count()
    print(f"{target}: {n} rows (source file has {expected})")

In [0]:
display(spark.table(f"{cat}.bronze.bronze_tower_master").limit(5))